# Spain 2027 EV Charging Network — Full EDA
## IE Sustainability Datathon 2026 · Iberdrola Challenge

| Section | Content |
|---|---|
| 1 | Road Network (M1) |
| 2 | BEV Fleet Projection (M3) |
| 3 | Charging Infrastructure Baseline (M2) |
| 4 | Cross-Dataset Synthesis |
| 5 | Grid Capacity — Endesa (R2) |
| 6 | Grid × Road Spatial Analysis |
| 7 | National Grid Context (REE) |
| 8 | Summary |

## 0. Setup

In [ ]:
import os, json, glob, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import requests
from scipy.optimize import curve_fit
from pyproj import Transformer

# Auto-detect base path (works whether CWD is notebooks/ or Datathon/)
for _d in ['.', '..']:
    if os.path.isdir(os.path.join(_d, 'Data')):
        BASE = os.path.abspath(_d)
        break
else:
    raise FileNotFoundError("Cannot find Data/ folder. Check working directory.")

RTIG_PATH     = f'{BASE}/Data/raw/road_routes_spain/carreteras_RTIG.geojson'
PARQUET_DIR   = f'{BASE}/Data/raw/ev_fleet_projections_datosgob/parquet'
M2_INTERURBAN = f'{BASE}/Data/interim/m2_charging_sites_interurban.csv'
M2_COVERAGE   = f'{BASE}/Data/interim/m2_road_coverage.csv'
M2_BASELINE   = f'{BASE}/Data/processed/m2_baseline.json'
M3_PROJECTION = f'{BASE}/Data/processed/m3_ev_projection.json'
ENDESA_CSV    = f'{BASE}/Data/external/grid_capacity_endesa/endesa_demanda_2026_03.csv'

print(f'BASE = {BASE}')
print()
for name, path in [('RTIG geojson', RTIG_PATH), ('M2 interurban', M2_INTERURBAN),
                   ('M2 coverage',  M2_COVERAGE), ('M2 baseline',  M2_BASELINE),
                   ('M3 projection',M3_PROJECTION),('Endesa CSV',   ENDESA_CSV)]:
    ok = os.path.exists(path)
    sz = f'{os.path.getsize(path)//1024} KB' if ok else ''
    print(f'  {"OK" if ok else "MISSING":7s}  {name:18s}  {sz}')

---
## 1. Road Network — Spain's Interurban Backbone (M1)

In [ ]:
roads = gpd.read_file(RTIG_PATH)

print(f'Segments  : {len(roads):,}')
print(f'Total km  : {roads["length_m"].sum()/1000:,.0f} km')
print(f'Unique roads: {roads["Carretera"].nunique():,}')
print()
print('By type:')
for tipo, grp in roads.groupby('Tipo_de_via'):
    print(f'  {tipo:35s}  {len(grp):4d} segments  {grp["length_m"].sum()/1000:,.0f} km')

In [ ]:
TYPE_COLORS = {
    'Autopista libre\\Autovía': '#2196F3',
    'Autopista peaje':            '#9C27B0',
    'Multicarril':                '#FF9800',
    'Carretera convencional':     '#4CAF50',
}
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for tipo, color in TYPE_COLORS.items():
    roads[roads['Tipo_de_via'] == tipo].plot(ax=ax, color=color, linewidth=0.7, label=tipo)
ax.legend(fontsize=7, loc='lower left')
ax.set_title('RTIG Interurban Road Network')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

ax = axes[1]
km = roads.groupby('Tipo_de_via')['length_m'].sum().div(1000).sort_values()
bars = ax.barh(km.index, km.values, color=[TYPE_COLORS.get(t,'#888') for t in km.index])
for bar, v in zip(bars, km.values):
    ax.text(v+50, bar.get_y()+bar.get_height()/2, f'{v:,.0f} km', va='center', fontsize=9)
ax.set_xlabel('km'); ax.set_title('Network Length by Road Type')
plt.tight_layout(); plt.show()

**Finding:** Spain's eligible interurban network covers ~29,050 km across 1,535 segments.
Free motorways (A-/AP-) dominate high-speed corridors; national roads (N-) add the secondary network connecting smaller towns.

---
## 2. BEV Fleet Projection 2027 (M3)

In [ ]:
# Load annual BEV data — parquets if available, pre-computed JSON if not
parquet_files = sorted(glob.glob(f'{PARQUET_DIR}/*.parquet'))

if parquet_files:
    print(f'Loading {len(parquet_files)} parquet files...')
    df = pd.concat([
        pd.read_parquet(f, columns=['COD_TIPO','CLAVE_TRAMITE','COD_PROPULSION_ITV','FECHA_TRAMITE'])
        for f in parquet_files
    ], ignore_index=True)
    df = df[(df['COD_TIPO']=='40') & (df['CLAVE_TRAMITE'].isin(['1','5','B']))].copy()
    df['year'] = pd.to_datetime(df['FECHA_TRAMITE'], errors='coerce').dt.year
    annual = pd.DataFrame({
        'total': df.groupby('year').size(),
        'bev':   df[df['COD_PROPULSION_ITV']=='6'].groupby('year').size()
    }).dropna().astype(int)
    annual['penetration'] = annual['bev'] / annual['total']
    DATA_SOURCE = 'DGT parquets'
elif os.path.exists(M3_PROJECTION):
    print('Loading from pre-computed projection JSON...')
    with open(M3_PROJECTION) as f: m3 = json.load(f)
    annual = pd.DataFrame({
        'total': {2019:960000,2020:851000,2021:860000,2022:900000,2023:950000},
        'bev':   {2019:2516,  2020:3625,  2021:9601,  2022:15882, 2023:27928},
    }); annual['penetration'] = annual['bev']/annual['total']
    DATA_SOURCE = 'pre-computed JSON'
else:
    print('Using hardcoded historical values')
    annual = pd.DataFrame({
        'total': {2019:960000,2020:851000,2021:860000,2022:900000,2023:950000},
        'bev':   {2019:2516,  2020:3625,  2021:9601,  2022:15882, 2023:27928},
    }); annual['penetration'] = annual['bev']/annual['total']
    DATA_SOURCE = 'hardcoded'

print(f'Source: {DATA_SOURCE}')
print()
print(f'{"Year":>6}  {"Total cars":>12}  {"BEV":>8}  {"Share":>7}')
for yr, row in annual[annual.index >= 2019].iterrows():
    print(f'{yr:>6}  {row.total:>12,}  {row.bev:>8,}  {row.penetration:>7.2%}')

In [ ]:
# Fit logistic penetration curve on 2019-2023 data
def logistic(t, L, k, t0):
    return L / (1 + np.exp(-k * (t - t0)))

if os.path.exists(M3_PROJECTION) and DATA_SOURCE == 'pre-computed JSON':
    with open(M3_PROJECTION) as f: m3 = json.load(f)
    p = m3['logistic_params']
    popt = (p['L'], p['k'], p['t0'])
    total_ev_projected_2027 = m3['total_ev_projected_2027']
    fleet_2023 = m3['fleet_baseline_2023']
    base_market = m3['base_market_size']
    proj_bev = {int(y): v for y, v in m3['annual_forecast'].items()}
    print('Loaded from pre-computed JSON')
else:
    fit_yrs = annual[annual.index >= 2019].index.astype(float).values
    fit_pct = annual[annual.index >= 2019]['penetration'].values
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        popt, _ = curve_fit(logistic, fit_yrs, fit_pct, p0=[0.5,0.5,2026],
                            bounds=([0.15,0.05,2024],[1.0,3.0,2035]), maxfev=10000)
    base_market = int(annual[annual.index.isin(range(2019,2024))]['total'].mean())
    proj_years  = range(2024, 2028)
    proj_pct    = {y: logistic(float(y), *popt) for y in proj_years}
    proj_bev    = {y: int(base_market * (1.02**(y-2023)) * proj_pct[y]) for y in proj_years}
    fleet_2023  = int(annual[annual.index <= 2023]['bev'].sum())
    total_ev_projected_2027 = fleet_2023 + sum(proj_bev.values())

    os.makedirs(f'{BASE}/Data/processed', exist_ok=True)
    with open(M3_PROJECTION, 'w') as f:
        json.dump({'total_ev_projected_2027': total_ev_projected_2027,
                   'fleet_baseline_2023': fleet_2023, 'base_market_size': base_market,
                   'logistic_params': {'L':round(float(popt[0]),4),'k':round(float(popt[1]),4),'t0':round(float(popt[2]),2)},
                   'annual_forecast': {str(y): proj_bev[y] for y in proj_bev}}, f, indent=2)
    print(f'Saved -> {M3_PROJECTION}')

print(f'\nLogistic params: max={popt[0]*100:.1f}%  k={popt[1]:.3f}  inflection={popt[2]:.1f}')
print(f'\nProjection:')
for y, n in proj_bev.items():
    print(f'  {y}: {n:>7,} new BEVs  ({logistic(float(y),*popt)*100:.1f}% market share)')
print(f'\nFleet baseline 2015-2023 : {fleet_2023:>8,}')
print(f'Projected new  2024-2027 : {sum(proj_bev.values()):>8,}')
print(f'total_ev_projected_2027  : {total_ev_projected_2027:>8,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
smooth = np.linspace(2015, 2030, 300)
ax.scatter(annual.index, annual['penetration']*100, color='#1565C0', s=60, zorder=4, label='Historical')
ax.plot(smooth, logistic(smooth, *popt)*100, color='#EF5350', lw=2, label='Logistic fit')
proj_x = list(proj_bev.keys())
ax.scatter(proj_x, [logistic(float(y),*popt)*100 for y in proj_x],
           color='#EF5350', s=60, marker='D', zorder=4, label='Projected')
ax.axvline(2023.5, color='grey', linestyle='--', alpha=0.4)
ax.set_xlabel('Year'); ax.set_ylabel('BEV share of new car registrations (%)')
ax.set_title('BEV Market Penetration — Logistic Fit'); ax.legend(fontsize=8)

ax = axes[1]
hist_bev = annual[annual.index >= 2019]['bev']
all_bev  = pd.concat([hist_bev, pd.Series(proj_bev)])
colors   = ['#42A5F5' if y <= 2023 else '#EF5350' for y in all_bev.index]
ax.bar(all_bev.index, all_bev.values, color=colors, alpha=0.85, width=0.7)
for x, v in zip(all_bev.index, all_bev.values):
    ax.text(x, v+200, f'{int(v):,}', ha='center', fontsize=7, rotation=45)
ax.axvline(2023.5, color='grey', linestyle='--', alpha=0.4)
ax.set_xlabel('Year'); ax.set_ylabel('New BEV registrations')
ax.set_title(f'BEV Registrations (2027 fleet: {total_ev_projected_2027:,})')
ax.legend(handles=[mpatches.Patch(color='#42A5F5',label='Historical'),
                   mpatches.Patch(color='#EF5350',label='Projected')], fontsize=8)
plt.tight_layout(); plt.show()

**Finding:** BEV market penetration follows an S-curve driven by falling battery costs and EU CO₂ mandates.
The logistic model projects accelerating growth through 2027 — unlike SARIMA which predicted decline.
`total_ev_projected_2027` is the key input for File 1.csv and demand gap calculations.

---
## 3. EV Charging Infrastructure Baseline (M2)

In [ ]:
df_sta = pd.read_csv(M2_INTERURBAN)
gdf_sta = gpd.GeoDataFrame(df_sta,
    geometry=gpd.points_from_xy(df_sta['longitude'], df_sta['latitude']), crs='EPSG:4326')

with open(M2_BASELINE) as f:
    n_stations = json.load(f)['total_existing_stations_baseline']

BINS   = [0, 50, 150, float('inf')]
LABELS = ['AC/DC slow (<50 kW)', 'DC fast (50-150 kW)', 'HPC (>=150 kW)']
df_sta['power_cat'] = pd.cut(df_sta['max_power_kw'], bins=BINS, labels=LABELS)
tier_counts = df_sta['power_cat'].value_counts().reindex(LABELS)

print(f'Interurban stations : {n_stations:,}')
print(f'Unique roads        : {df_sta["Carretera"].nunique():,}')
print(f'Avg connectors/site : {df_sta["n_refill_points"].mean():.1f}')
print()
print('Power tier breakdown:')
for label, count in tier_counts.items():
    bar = '#' * int(count / n_stations * 40)
    print(f'  {label:22s}: {count:>5,} ({count/n_stations*100:>5.1f}%)  {bar}')

In [ ]:
TIER_COLORS = ['#FFC107', '#FF5722', '#B71C1C']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

ax = axes[0]
roads.plot(ax=ax, color='lightgrey', linewidth=0.4, alpha=0.6)
for label, color in zip(LABELS, TIER_COLORS):
    gdf_sta[gdf_sta['power_cat']==label].plot(ax=ax, markersize=2, color=color, alpha=0.7)
ax.legend(handles=[mpatches.Patch(color=c, label=f'{l} ({tier_counts[l]:,})')
                   for l,c in zip(LABELS,TIER_COLORS)], fontsize=7, loc='lower left')
ax.set_title('Interurban Stations by Power Tier')

ax = axes[1]
tier_counts.plot(kind='barh', ax=ax, color=TIER_COLORS[::-1])
ax.set_xlabel('Number of stations')
ax.set_title('Power Tier Distribution')
for i, v in enumerate(tier_counts[::-1]):
    ax.text(v+10, i, f'{v:,} ({v/n_stations*100:.0f}%)', va='center', fontsize=9)

ax = axes[2]
top_ops = df_sta['operator'].value_counts().head(10)[::-1]
top_ops.plot(kind='barh', ax=ax, color='#1565C0', alpha=0.8)
ax.set_xlabel('Interurban sites'); ax.set_title('Top 10 Operators')
for i, v in enumerate(top_ops):
    ax.text(v+1, i, str(v), va='center', fontsize=8)

plt.tight_layout(); plt.show()

**Finding — Quality gap, not coverage gap.**
84% of interurban stations are below the 150 kW HPC threshold needed for sub-30-minute highway stops.
The network has physical presence (3,679 sites, 281 roads) but is dominated by AC-only and DC fast chargers
inadequate for highway pit-stops. Iberdrola leads the interurban segment.

---
## 4. Cross-Dataset Synthesis — The 2027 Demand Gap

In [ ]:
ev_per_station    = total_ev_projected_2027 / n_stations
benchmark_low, benchmark_high = 20, 30

print('=' * 52)
print('  SUPPLY vs. DEMAND — 2027 PROJECTION')
print('=' * 52)
print(f'  Projected BEV fleet (2027)   : {total_ev_projected_2027:>8,}')
print(f'  Existing interurban stations : {n_stations:>8,}')
print(f'  Current EVs per station      : {ev_per_station:>8.0f}x')
print(f'  Industry benchmark           : {benchmark_low}–{benchmark_high}x')
print(f'  Station deficit (20:1 bench) : {total_ev_projected_2027//benchmark_low - n_stations:>8,}')
print(f'  Station deficit (30:1 bench) : {total_ev_projected_2027//benchmark_high - n_stations:>8,}')
print('=' * 52)

In [ ]:
coverage = pd.read_csv(M2_COVERAGE)
roads_vis = roads[['Carretera','geometry']].copy().reset_index(drop=True)
roads_vis['nearest_km'] = coverage['nearest_station_m'].values / 1000
roads_vis['has_gap']    = coverage['has_gap'].values

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ax = axes[0]
norm = mcolors.Normalize(vmin=0, vmax=50)
cmap = plt.get_cmap('RdYlGn_r')
for _, row in roads_vis.iterrows():
    gpd.GeoSeries([row.geometry]).plot(ax=ax, color=[cmap(norm(min(row.nearest_km,50)))],
                                        linewidth=0.7, alpha=0.9)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.6)
cbar.set_label('Distance to nearest station (km)')
roads_vis[roads_vis['has_gap']].plot(ax=ax, color='#B71C1C', linewidth=3,
                                      zorder=5, label='Gap >50 km (N-502, N-502A)')
ax.legend(fontsize=8, loc='lower left')
ax.set_title('Road Coverage — Distance to Nearest Station')

ax = axes[1]
vals  = [n_stations, total_ev_projected_2027//benchmark_low, total_ev_projected_2027//benchmark_high]
cats  = ['Existing
stations','Needed
(20:1)','Needed
(30:1)']
bars  = ax.bar(cats, vals, color=['#42A5F5','#EF5350','#FF8A65'], width=0.5, alpha=0.85)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
            f'{v:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Interurban stations'); ax.set_title('Station Demand vs. Supply (2027)')

plt.tight_layout(); plt.show()

**Finding:** 99.6% of road segments are within 50 km of an existing station — only N-502 and N-502A are true dark corridors.
The gap is not geographic coverage but **station quality and density**: the current ~60:1 EV-to-station ratio needs to reach 20–30:1, requiring 3,700–7,300 new stations.

---
## 5. Grid Capacity — Endesa Distribution Zone (R2)

In [ ]:
df_raw = pd.read_csv(ENDESA_CSV, sep=';', decimal=',', encoding='utf-8-sig')
df_raw = df_raw.rename(columns={
    'Coordenada UTM X': 'utm_x', 'Coordenada UTM Y': 'utm_y',
    'Nivel de Tensión (kV)': 'kv',
    'Capacidad firme disponible (MW)': 'available_mw',
    'Nombre Subestación': 'nombre', 'Comunidad Autónoma': 'ccaa',
    'Subestación': 'sub_id', 'Provincia.1': 'provincia',
})
for col in ['utm_x','utm_y','kv','available_mw']:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# Aggregate: max available capacity per substation across voltage levels
sub = df_raw.groupby('sub_id').agg(
    utm_x=('utm_x','first'), utm_y=('utm_y','first'),
    nombre=('nombre','first'), ccaa=('ccaa','first'), provincia=('provincia','first'),
    max_kv=('kv','max'), available_mw=('available_mw','max'),
).reset_index().dropna(subset=['utm_x','utm_y'])

# UTM ETRS89 Zone 30N → WGS84
t = Transformer.from_crs('EPSG:25830','EPSG:4326', always_xy=True)
sub['longitude'], sub['latitude'] = t.transform(sub['utm_x'].values, sub['utm_y'].values)
valid = sub['longitude'].between(-10,5) & sub['latitude'].between(35,44)
gdf_sub = gpd.GeoDataFrame(sub[valid].reset_index(drop=True),
    geometry=gpd.points_from_xy(sub[valid]['longitude'], sub[valid]['latitude']),
    crs='EPSG:4326')

print(f'Substations (mainland Spain): {len(gdf_sub):,}')
print(f'CCAAs covered: {gdf_sub["ccaa"].nunique()}')
print(f'Voltage levels: {sorted(df_raw["kv"].dropna().unique().astype(int))}')
print(f'Available capacity — median: {gdf_sub["available_mw"].median():.1f} MW, '
      f'max: {gdf_sub["available_mw"].max():.1f} MW')
print(f'Zero-capacity substations: {(gdf_sub["available_mw"]==0).sum()} '
      f'({(gdf_sub["available_mw"]==0).mean()*100:.0f}%)')

In [ ]:
# Classify grid status (1 HPC = 0.15 MW; 10-charger hub = 1.5 MW)
def classify(mw):
    return 'Sufficient' if mw >= 5 else ('Moderate' if mw > 0 else 'Congested')

gdf_sub['grid_status'] = gdf_sub['available_mw'].apply(classify)
STATUS_COLORS = {'Sufficient':'#4CAF50', 'Moderate':'#FF9800', 'Congested':'#F44336'}
counts = gdf_sub['grid_status'].value_counts().reindex(['Sufficient','Moderate','Congested'])

print('Grid Status  | Substations |   %  | Total avail. MW')
print('─' * 55)
for s in ['Sufficient','Moderate','Congested']:
    n = counts[s]; pct = n/len(gdf_sub)*100
    mw = gdf_sub[gdf_sub['grid_status']==s]['available_mw'].sum()
    print(f'{s:<12} | {n:>11,} | {pct:>4.0f}% | {mw:>14.1f} MW')
print('─' * 55)
print(f'{"TOTAL":<12} | {len(gdf_sub):>11,} |      | '
      f'{gdf_sub["available_mw"].sum():>14.1f} MW')

# Save for downstream use
os.makedirs(f'{BASE}/Data/interim', exist_ok=True)
gdf_sub[['sub_id','nombre','provincia','ccaa','longitude','latitude',
         'max_kv','available_mw','grid_status']].to_csv(
    f'{BASE}/Data/interim/grid_endesa_substations.csv', index=False)
print(f'\nSaved grid_endesa_substations.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

ax = axes[0]
roads.plot(ax=ax, color='#E0E0E0', linewidth=0.3, alpha=0.6)
for status, color in STATUS_COLORS.items():
    gdf_sub[gdf_sub['grid_status']==status].plot(
        ax=ax, markersize=6 if status=='Sufficient' else 3, color=color, alpha=0.85, zorder=3)
ax.legend(handles=[mpatches.Patch(color=c, label=f'{s} ({counts[s]:,})')
                   for s,c in STATUS_COLORS.items()], fontsize=9, loc='lower left')
ax.set_title('Endesa Substations — Grid Status')

ax = axes[1]
for s, c in STATUS_COLORS.items():
    vals = gdf_sub[gdf_sub['grid_status']==s]['available_mw']
    vals = vals[vals > 0]
    if len(vals): ax.hist(vals, bins=25, color=c, alpha=0.75, label=f'{s} (n={len(vals)})')
ax.axvline(0.15, color='grey', linestyle=':', lw=1.2, label='1 HPC (0.15 MW)')
ax.axvline(1.5,  color='#1565C0', linestyle=':', lw=1.2, label='10-hub (1.5 MW)')
ax.set_xlabel('Available Capacity (MW)'); ax.set_ylabel('Substations')
ax.set_title('Capacity Distribution (non-zero)'); ax.legend(fontsize=8); ax.set_xlim(0, 30)

ax = axes[2]
ccaa_s = (gdf_sub.groupby(['ccaa','grid_status']).size().unstack(fill_value=0)
          .reindex(columns=['Sufficient','Moderate','Congested'], fill_value=0))
ccaa_s.index = ccaa_s.index.str.replace(r'^\d+ - ','', regex=True)
ccaa_s.plot(kind='barh', ax=ax, color=[STATUS_COLORS[s] for s in ccaa_s.columns],
            stacked=True, width=0.7, alpha=0.85)
ax.set_xlabel('Substations'); ax.set_title('Grid Status by CCAA (Endesa Zone)')
ax.legend(fontsize=8)

plt.suptitle('Endesa Distribution Grid — Capacity Analysis', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

**Finding:** ~65% of Endesa substations have zero available firm capacity — already fully booked.
Only ~15% have ≥5 MW (enough for a full HPC hub). Grid saturation, not geography, is the primary deployment constraint in Andalucía, Aragón, and Cataluña.

---
## 6. Grid Status Assigned to Interurban Stations

In [ ]:
# Match each interurban station to its nearest Endesa substation
sub_m = gdf_sub[['sub_id','available_mw','grid_status','geometry']].to_crs('EPSG:25830')
sta_m = gdf_sta[['site_id','Carretera','max_power_kw','geometry']].to_crs('EPSG:25830')

nearest = gpd.sjoin_nearest(sta_m, sub_m, how='left', distance_col='dist_to_sub_m')
nearest = nearest[~nearest.index.duplicated(keep='first')]

# Only assign Endesa status within 25 km (beyond = i-DE or Viesgo territory)
nearest['in_endesa_zone'] = nearest['dist_to_sub_m'] <= 25_000
nearest['assigned_status'] = nearest.apply(
    lambda r: r['grid_status'] if r['in_endesa_zone'] else 'i-DE/Viesgo (pending)', axis=1)

endesa = nearest[nearest['in_endesa_zone']]
gs = endesa['assigned_status'].value_counts().reindex(['Sufficient','Moderate','Congested'])

print(f'Total interurban stations    : {len(nearest):,}')
print(f'In Endesa zone (<=25 km sub) : {len(endesa):,}')
print(f'Other zones (i-DE/Viesgo)    : {(~nearest["in_endesa_zone"]).sum():,}')
print()
print('Grid status — Endesa zone stations:')
for s, n in gs.items():
    print(f'  {s:<12}: {n:>5,}  ({n/len(endesa)*100:.0f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
ALL_COLORS = {**STATUS_COLORS, 'i-DE/Viesgo (pending)':'#90A4AE'}

ax = axes[0]
roads.plot(ax=ax, color='#E0E0E0', linewidth=0.3, alpha=0.6)
nearest_wgs = nearest.copy()
nearest_wgs.geometry = nearest_wgs.geometry.to_crs('EPSG:4326')
for status, color in ALL_COLORS.items():
    sub_n = nearest_wgs[nearest_wgs['assigned_status']==status]
    if len(sub_n):
        gpd.GeoDataFrame(sub_n, geometry=sub_n.geometry, crs='EPSG:4326').plot(
            ax=ax, markersize=4 if status!='i-DE/Viesgo (pending)' else 1.5,
            color=color, alpha=0.85, zorder=3)
ax.legend(handles=[mpatches.Patch(color=c,
    label=f'{s} ({(nearest["assigned_status"]==s).sum()})')
    for s,c in ALL_COLORS.items()], fontsize=8, loc='lower left')
ax.set_title('Interurban Stations — Assigned Grid Status')

ax = axes[1]
dist_km = nearest['dist_to_sub_m'] / 1000
ax.hist(dist_km[dist_km<=60], bins=60, color='#1565C0', alpha=0.8, edgecolor='white', linewidth=0.2)
ax.axvline(25, color='#F44336', linestyle='--', lw=1.5, label='25 km zone boundary')
ax.text(26, ax.get_ylim()[1]*0.85, f'{(dist_km<=25).mean()*100:.0f}% within 25 km',
        color='#F44336', fontsize=9)
ax.set_xlabel('Distance to nearest Endesa substation (km)')
ax.set_ylabel('Stations'); ax.set_title('Station–Substation Proximity')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

**Finding:** Stations in the Endesa zone near Congested substations cluster on the A-4, A-7, and A-2 —
Spain's busiest motorways. These corridors have the highest EV demand AND the most saturated grid.
Any new deployment on these corridors must include a grid upgrade plan.

---
## 7. National Grid Context (REE apidatos.ree.es)

In [ ]:
REE = 'https://apidatos.ree.es'

def ree_get(endpoint, **params):
    try:
        r = requests.get(f'{REE}{endpoint}', params=params, timeout=15)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f'  API error: {e}'); return None

# National demand trend 2019-2024
print('Fetching national demand trend...')
data = ree_get('/es/datos/demanda/demanda-real',
               start_date='2019-01-01T00:00', end_date='2024-12-31T23:59', time_trunc='year')

df_nat = None
if data:
    vals = data.get('data',{}).get('attributes',{}).get('values',[])
    if vals:
        df_nat = pd.DataFrame([{'year': int(v['datetime'][:4]), 'twh': v['value']/1000} for v in vals])
        print(df_nat.to_string(index=False))

# CCAA demand breakdown
print('\nFetching CCAA demand 2024...')
data_ccaa = ree_get('/es/datos/demanda/demanda-real',
                    start_date='2024-01-01T00:00', end_date='2024-12-31T23:59',
                    time_trunc='year', geo_trunc='electric_system', geo_limit='ccaa')
df_ccaa = None
if data_ccaa:
    records = [{'ccaa': item['attributes']['title'],
                'twh': item['attributes']['values'][0]['value']/1000}
               for item in data_ccaa.get('included',[])
               if item.get('attributes',{}).get('values')]
    if records:
        df_ccaa = pd.DataFrame(records).sort_values('twh', ascending=False)
        print(f'Got {len(df_ccaa)} CCAAs')

In [ ]:
ENDESA_ZONE = {'Andalucía','Cataluña','Aragón','Extremadura','Canarias','Illes Balears'}
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
if df_nat is not None:
    bars = ax.bar(df_nat['year'], df_nat['twh'], color='#1565C0', alpha=0.85, width=0.6)
    for bar, v in zip(bars, df_nat['twh']):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{v:.0f}', ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('Year'); ax.set_ylabel('TWh')
    ax.set_title('Spain National Electricity Demand 2019–2024')
    ev_add = total_ev_projected_2027 * 3000 * 0.2 / 1e9  # rough estimate
    ax.annotate(f'EV charging 2027\n~+{ev_add:.2f} TWh (<0.1%)',
                xy=(2023.7, df_nat['twh'].iloc[-1]),
                xytext=(2020.5, df_nat['twh'].iloc[-1]*0.92),
                arrowprops=dict(arrowstyle='->', color='#F44336'),
                fontsize=8, color='#C62828')
else:
    ax.text(0.5,0.5,'API unavailable',ha='center',va='center',transform=ax.transAxes,color='grey')
    ax.axis('off')

ax = axes[1]
if df_ccaa is not None:
    df_plot = df_ccaa.sort_values('twh').tail(15)
    colors  = ['#1565C0' if any(e.lower() in r['ccaa'].lower() for e in ENDESA_ZONE)
               else '#90A4AE' for _,r in df_plot.iterrows()]
    bars = ax.barh(df_plot['ccaa'], df_plot['twh'], color=colors, alpha=0.85)
    for bar, v in zip(bars, df_plot['twh']):
        ax.text(v+0.1, bar.get_y()+bar.get_height()/2, f'{v:.0f}', va='center', fontsize=8)
    ax.set_xlabel('TWh (2024)'); ax.set_title('Electricity Demand by CCAA\n(Blue = Endesa zone)')
    ax.legend(handles=[mpatches.Patch(color='#1565C0',label='Endesa zone'),
                       mpatches.Patch(color='#90A4AE',label='i-DE/Viesgo zone')], fontsize=9)
else:
    ax.text(0.5,0.5,'CCAA data unavailable\n(public API)',
            ha='center',va='center',transform=ax.transAxes,color='grey'); ax.axis('off')
plt.tight_layout(); plt.show()

**Finding:** Spain's total electricity demand is ~250 TWh/year.
All 220,000 projected EVs charging on interurban roads adds roughly **+0.1 TWh (<0.1%)** nationally.
The grid is not under national-level stress from EVs — the constraint is hyper-local: **which specific substation** serves each charging site.

---
## 8. Key Findings Summary

In [ ]:
print('=' * 68)
print('  SPAIN 2027 EV CHARGING NETWORK — EDA SUMMARY')
print('=' * 68)
print()
print('  ROAD NETWORK (M1)')
print(f'    Eligible segments  : {len(roads):,}  ({roads["length_m"].sum()/1000:,.0f} km)')
print()
print('  BEV FLEET PROJECTION (M3)')
print(f'    Historical fleet 2015-2023    : {fleet_2023:>8,} BEVs')
print(f'    Projected new    2024-2027    : {sum(proj_bev.values()):>8,} BEVs')
print(f'    total_ev_projected_2027       : {total_ev_projected_2027:>8,} BEVs')
print()
print('  CHARGING BASELINE (M2)')
print(f'    National sites                : {12074:>8,}')
print(f'    Interurban sites (<=500m RTIG): {n_stations:>8,}  ({n_stations/12074*100:.1f}%)')
print(f'    HPC sites (>=150 kW)          : {tier_counts["HPC (>=150 kW)"]:>8,}  ({tier_counts["HPC (>=150 kW)"]/n_stations*100:.1f}%)')
print(f'    Roads with coverage gap >50km : {"N-502, N-502A (130 km)":>25s}')
print()
print('  DEMAND GAP')
print(f'    Current EV-to-station ratio   : {ev_per_station:>8.0f}x')
print(f'    Target benchmark              : {"20-30x":>8s}')
print(f'    Station deficit (20:1)        : {total_ev_projected_2027//20 - n_stations:>8,}')
print(f'    Station deficit (30:1)        : {total_ev_projected_2027//30 - n_stations:>8,}')
print()
print('  GRID CAPACITY (R2 — ENDESA ZONE)')
print(f'    Substations analysed          : {len(gdf_sub):>8,}')
print(f'    Congested (0 MW available)    : {counts["Congested"]:>8,}  ({counts["Congested"]/len(gdf_sub)*100:.0f}%)')
print(f'    Sufficient (>=5 MW)           : {counts["Sufficient"]:>8,}  ({counts["Sufficient"]/len(gdf_sub)*100:.0f}%)')
print()
print('  CORE FINDINGS')
print('  1. Coverage is solved. Quality is not.')
print('     99.6% roads covered; 84% stations below 150 kW HPC threshold.')
print('  2. Grid saturation is the main deployment constraint.')
print('     65% of Endesa substations have ZERO available capacity.')
print('  3. EV demand is <0.1% of national grid — local substations are the bottleneck.')
print('=' * 68)

### Outputs for Modelling Team

| File | Contents | Used by |
|---|---|---|
| `m2_charging_sites_interurban.csv` | 3,679 existing stations + road + power tier | M4 placement |
| `m2_road_coverage.csv` | Per-segment nearest station distance + gap flag | M4 prioritisation |
| `m2_baseline.json` | `total_existing_stations_baseline = 3679` | File 1.csv |
| `m3_ev_projection.json` | `total_ev_projected_2027`, logistic params | File 1.csv |
| `grid_endesa_substations.csv` | 1,034 substations + `grid_status` (Endesa zone) | M4 grid assignment |

**`grid_status` thresholds:** Sufficient ≥5 MW · Moderate >0 MW · Congested =0 MW  
**Zone boundary:** Endesa = nearest substation ≤25 km · i-DE/Viesgo = pending R1/R3 data